# EN4554 Assignment 2: Neural Networks and CNNs on CIFAR-10

**Name:** _your name_
**Index number:** _your index number_
**GitHub:** https://github.com/HimethW/EN4554-Assignment_2-on-CNNs

Ground rules used throughout:

* The 50,000 CIFAR-10 training images are split once, with a fixed seed, into **45,000 training** and **5,000 validation** images.
* Inputs are normalised per channel with the mean and standard deviation of the **45k training images only**.
* Random seeds are fixed, so runs that are compared start from the same weights.
* The **test set is used exactly once**, at the end of Task 4. Every other accuracy in this report is a validation accuracy.

## 0. Setup

`QUICK_RUN = True` trains every experiment for one epoch and skips the test-set cell. It only checks that the notebook runs end to end (e.g. on a Colab T4); the results in this report use `QUICK_RUN = False`.

In [ ]:
import math
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

QUICK_RUN = False
SEED = 0
DATA_DIR = "./data"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def n_epochs(n):
    return 1 if QUICK_RUN else n


set_seed()
torch.backends.cudnn.benchmark = True

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.prop_cycle": plt.cycler(color=["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
                                         "#e87ba4", "#008300", "#6250d6", "#e34948"]),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#c3c2b7",
    "axes.grid": True,
    "grid.color": "#e1e0d9",
    "grid.linewidth": 0.6,
    "lines.linewidth": 2,
    "legend.frameon": False,
})

gpu = torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU"
print(f"PyTorch {torch.__version__} on {gpu}, QUICK_RUN = {QUICK_RUN}")

### Data

The training images are shuffled once with a fixed seed: the first 45,000 form the training split and the remaining 5,000 the validation split. The per-channel mean and standard deviation are computed on the 45k training images only, so nothing about the held-out images leaks into training.

Both splits are kept as tensors on the GPU (about 0.6 GB). Batching, and later data augmentation, then run on the GPU, which is much faster than a `DataLoader` that processes one PIL image at a time. The test set is only loaded by `load_test_set()`, in Task 4.

In [ ]:
cifar = torchvision.datasets.CIFAR10(DATA_DIR, train=True, download=True)
CLASSES = cifar.classes
images = torch.from_numpy(cifar.data).permute(0, 3, 1, 2).float() / 255
labels = torch.tensor(cifar.targets)

perm = torch.randperm(len(images), generator=torch.Generator().manual_seed(SEED))
train_idx, val_idx = perm[:45_000], perm[45_000:]

MEAN = images[train_idx].mean(dim=(0, 2, 3))
STD = images[train_idx].std(dim=(0, 2, 3))


def normalize(x):
    return (x - MEAN.view(1, 3, 1, 1)) / STD.view(1, 3, 1, 1)


X_train, y_train = normalize(images[train_idx]).to(device), labels[train_idx].to(device)
X_val, y_val = normalize(images[val_idx]).to(device), labels[val_idx].to(device)
del images, labels


def load_test_set():  # used once, in Task 4
    test = torchvision.datasets.CIFAR10(DATA_DIR, train=False, download=True)
    x = torch.from_numpy(test.data).permute(0, 3, 1, 2).float() / 255
    return normalize(x).to(device), torch.tensor(test.targets).to(device)


def batches(X, y, batch_size, shuffle=False, gen=None):
    order = torch.randperm(len(X), generator=gen).to(X.device) if shuffle else torch.arange(len(X), device=X.device)
    for i in range(0, len(X), batch_size):
        idx = order[i:i + batch_size]
        yield X[idx], y[idx]


print("train", tuple(X_train.shape), "| validation", tuple(X_val.shape))
print("mean (R, G, B):", MEAN.numpy().round(4), "| std (R, G, B):", STD.numpy().round(4))
print("validation images per class:", torch.bincount(y_val.cpu(), minlength=10).tolist())

In [ ]:
fig, axes = plt.subplots(1, 10, figsize=(12, 1.6))
for c, ax in enumerate(axes):
    img = X_train[(y_train == c).nonzero()[0, 0]].cpu() * STD.view(3, 1, 1) + MEAN.view(3, 1, 1)
    ax.imshow(img.permute(1, 2, 0).clamp(0, 1))
    ax.set_title(CLASSES[c], fontsize=9)
    ax.axis("off")
plt.show()

# Task 1: Backpropagation

## 1(a) Linear, ReLU and softmax cross-entropy with explicit backward passes

Every layer has a `forward(x)` that caches what the backward pass needs, and a `backward(dout)` that takes $\partial L/\partial(\text{output})$, stores the gradients of the layer's own parameters and returns $\partial L/\partial(\text{input})$. The parameters are plain tensors (no `requires_grad`), so autograd is never involved. A mini-batch has $N$ rows.

**Linear**, $y = xW^\top + b$ with $W \in \mathbb{R}^{d_\text{out}\times d_\text{in}}$. Since $y_{nj} = \sum_i x_{ni}W_{ji} + b_j$, the chain rule gives

$$
\frac{\partial L}{\partial W} = \Big(\frac{\partial L}{\partial y}\Big)^{\!\top} x, \qquad
\frac{\partial L}{\partial b} = \sum_{n=1}^{N} \frac{\partial L}{\partial y_{n}}, \qquad
\frac{\partial L}{\partial x} = \frac{\partial L}{\partial y}\, W .
$$

**ReLU**, $y = \max(0, x)$: $\;\partial L/\partial x = \partial L/\partial y \odot \mathbb{1}[x > 0]$.

**Softmax cross-entropy.** With $\log p_{nk} = z_{nk} - \log\sum_j e^{z_{nj}}$ and $L = -\frac{1}{N}\sum_n \log p_{n y_n}$,

$$
\frac{\partial L}{\partial z_{nk}} = \frac{1}{N}\big(p_{nk} - \mathbb{1}[k = y_n]\big),
$$

the predicted probabilities minus the one-hot label. Treating softmax and cross-entropy as one layer gives this simple gradient and lets us compute $\log p$ with the log-sum-exp trick (subtract $\max_j z_{nj}$ first so `exp` cannot overflow).

**Initialisation.** $W \sim \mathcal{N}(0, \sigma^2)$ with $\sigma^2 = 2/d_\text{in}$ (Kaiming/He, suited to ReLU) or $\sigma^2 = 2/(d_\text{in}+d_\text{out})$ (Xavier/Glorot, suited to sigmoid; used in 1(d)). Biases start at zero.

In [ ]:
class Linear:
    def __init__(self, n_in, n_out, init="kaiming", dtype=torch.float32):
        std = math.sqrt(2 / n_in) if init == "kaiming" else math.sqrt(2 / (n_in + n_out))
        self.W = (torch.randn(n_out, n_in, dtype=dtype) * std).to(device)
        self.b = torch.zeros(n_out, dtype=dtype, device=device)

    def forward(self, x):
        self.x = x
        return x @ self.W.T + self.b

    def backward(self, dout):
        self.dW = dout.T @ self.x
        self.db = dout.sum(dim=0)
        return dout @ self.W


class ReLU:
    def forward(self, x):
        self.mask = x > 0
        return x * self.mask

    def backward(self, dout):
        return dout * self.mask


class SoftmaxCrossEntropy:
    def forward(self, logits, y):
        z = logits - logits.max(dim=1, keepdim=True).values  # log-sum-exp trick
        log_p = z - z.exp().sum(dim=1, keepdim=True).log()
        self.p, self.y = log_p.exp(), y
        self.rows = torch.arange(len(y), device=y.device)
        return -log_p[self.rows, y].mean()

    def backward(self):
        grad = self.p.clone()
        grad[self.rows, self.y] -= 1
        return grad / len(self.y)


class MLP:
    def __init__(self, sizes, activation=ReLU, init="kaiming", dtype=torch.float32):
        self.linears = [Linear(a, b, init, dtype) for a, b in zip(sizes[:-1], sizes[1:])]
        self.layers = []
        for lin in self.linears:
            self.layers += [lin, activation()]
        self.layers.pop()  # the last layer outputs logits
        self.criterion = SoftmaxCrossEntropy()

    def forward(self, x):
        for layer in self.layers:
            x = layer.forward(x)
        return x

    def loss(self, x, y):
        return self.criterion.forward(self.forward(x), y)

    def backward(self):
        grad = self.criterion.backward()
        for layer in reversed(self.layers):
            grad = layer.backward(grad)

    def params(self):
        return [p for lin in self.linears for p in (lin.W, lin.b)]

    def grads(self):
        return [g for lin in self.linears for g in (lin.dW, lin.db)]

A quick sanity check: one forward and backward pass of a 3072 → 512 → 256 → 10 MLP on 256 flattened training images.

In [ ]:
set_seed()
mlp = MLP([3072, 512, 256, 10])
loss = mlp.loss(X_train[:256].flatten(1), y_train[:256])
mlp.backward()

print(f"initial loss {loss.item():.3f} (ln 10 = {math.log(10):.3f})")
print("autograd used:", loss.requires_grad)
print("parameters:", sum(p.numel() for p in mlp.params()))
print("gradient shapes match:", all(p.shape == g.shape for p, g in zip(mlp.params(), mlp.grads())))

_INTERPRETATION 1A_

## 1(b) Gradient check

The manual gradients are compared with two independent references:

1. **Autograd.** The same weights are copied into an `nn.Sequential` of `nn.Linear`/`nn.ReLU` layers, the loss is computed with `F.cross_entropy`, and `backward()` gives the reference gradients for every entry.
2. **Centred finite differences**, $\;\dfrac{\partial L}{\partial \theta_i} \approx \dfrac{L(\theta + h e_i) - L(\theta - h e_i)}{2h}$ with $h = 10^{-6}$, whose error is $O(h^2)$. This needs two forward passes per weight, so it is evaluated on 20 random entries of each parameter tensor.

For each parameter tensor we report the relative error $\;\|g - \hat g\|_2 \,/\, \max(\|g\|_2, \|\hat g\|_2)$ and whether `torch.allclose` (default tolerances, `rtol=1e-5`, `atol=1e-8`) accepts the match. The check runs in float64: a finite difference subtracts two almost equal losses, and float32 rounding (about $10^{-7}$ of the loss) divided by $2h$ would swamp the gradient.

In [ ]:
def rel_error(a, b):
    return ((a - b).norm() / torch.maximum(a.norm(), b.norm())).item()


def autograd_grads(mlp, x, y):
    layers = []
    for lin in mlp.linears:
        layer = nn.Linear(lin.W.shape[1], lin.W.shape[0]).to(device, lin.W.dtype)
        with torch.no_grad():
            layer.weight.copy_(lin.W)
            layer.bias.copy_(lin.b)
        layers += [layer, nn.ReLU()]
    net = nn.Sequential(*layers[:-1])
    F.cross_entropy(net(x), y).backward()
    return [p.grad for p in net.parameters()]


def finite_diff_grads(mlp, x, y, n_entries=20, h=1e-6):
    estimates = []
    for p in mlp.params():
        flat = p.view(-1)
        idx = torch.randperm(flat.numel())[:n_entries]
        est = torch.empty(len(idx), dtype=p.dtype)
        for k, i in enumerate(idx):
            old = flat[i].item()
            flat[i] = old + h
            loss_plus = mlp.loss(x, y).item()
            flat[i] = old - h
            loss_minus = mlp.loss(x, y).item()
            flat[i] = old
            est[k] = (loss_plus - loss_minus) / (2 * h)
        estimates.append((idx, est))
    return estimates

In [ ]:
set_seed()
mlp = MLP([3072, 512, 256, 10], dtype=torch.float64)
x, y = X_train[:64].flatten(1).double(), y_train[:64]
mlp.loss(x, y)
mlp.backward()

auto = autograd_grads(mlp, x, y)
numeric = finite_diff_grads(mlp, x, y)

print(f"{'param':<7}{'shape':<14}{'vs autograd':>13}{'allclose':>10}{'vs finite diff':>17}{'allclose':>10}")
for i, (g, a, (idx, est)) in enumerate(zip(mlp.grads(), auto, numeric)):
    name = ("W", "b")[i % 2] + str(i // 2 + 1)
    g_sub = g.view(-1)[idx].cpu()
    print(f"{name:<7}{str(tuple(g.shape)):<14}{rel_error(g, a):>13.1e}{str(torch.allclose(g, a)):>10}"
          f"{rel_error(g_sub, est):>17.1e}{str(torch.allclose(g_sub, est)):>10}")

_INTERPRETATION 1B_

## 1(c) Training the MLP with SGD

The 3072 → 512 → 256 → 10 ReLU MLP is trained on flattened images with plain mini-batch SGD, $\theta \leftarrow \theta - \eta\, \nabla_\theta L$, using batch size 128 and learning rate $\eta = 0.01$ for 30 epochs. After every epoch the loss and accuracy are measured on the full training and validation splits.

In [ ]:
def evaluate(mlp, X, y, batch_size=1000):
    loss_sum, correct = 0.0, 0
    for xb, yb in batches(X, y, batch_size):
        logits = mlp.forward(xb.flatten(1))
        loss_sum += mlp.criterion.forward(logits, yb).item() * len(yb)
        correct += (logits.argmax(1) == yb).sum().item()
    return loss_sum / len(X), correct / len(X)


def train_sgd(mlp, epochs, lr, batch_size=128):
    gen = torch.Generator().manual_seed(SEED)
    hist = {"step_loss": [], "train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    for epoch in range(epochs):
        for xb, yb in batches(X_train, y_train, batch_size, shuffle=True, gen=gen):
            loss = mlp.loss(xb.flatten(1), yb)
            mlp.backward()
            for p, g in zip(mlp.params(), mlp.grads()):
                p -= lr * g
            hist["step_loss"].append(loss.item())

        for split, X, y in [("train", X_train, y_train), ("val", X_val, y_val)]:
            split_loss, split_acc = evaluate(mlp, X, y)
            hist[f"{split}_loss"].append(split_loss)
            hist[f"{split}_acc"].append(split_acc)
        print(f"epoch {epoch + 1:2d}  train loss {hist['train_loss'][-1]:.3f}  "
              f"val loss {hist['val_loss'][-1]:.3f}  val acc {hist['val_acc'][-1]:.3f}")
    return hist


set_seed()
mlp = MLP([3072, 512, 256, 10])
hist = train_sgd(mlp, n_epochs(30), lr=0.01)

In [ ]:
epochs = np.arange(1, len(hist["val_loss"]) + 1)
steps = np.arange(1, len(hist["step_loss"]) + 1) / (len(hist["step_loss"]) / len(epochs))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(steps, hist["step_loss"], color="#b7d3f6", lw=0.6, label="mini-batch (train)")
ax1.plot(epochs, hist["train_loss"], label="train")
ax1.plot(epochs, hist["val_loss"], label="validation")
ax1.set(xlabel="epoch", ylabel="cross-entropy loss", title="Loss")
ax1.legend()

ax2.plot(epochs, np.array(hist["train_acc"]) * 100, label="train")
ax2.plot(epochs, np.array(hist["val_acc"]) * 100, label="validation")
ax2.set(xlabel="epoch", ylabel="accuracy (%)", title="Accuracy")
ax2.legend()
plt.tight_layout()
plt.show()

best = int(np.argmax(hist["val_acc"]))
print(f"final validation accuracy: {hist['val_acc'][-1]:.2%}")
print(f"best validation accuracy:  {hist['val_acc'][best]:.2%} (epoch {best + 1})")

_INTERPRETATION 1C_

## 1(d) Per-layer gradient norms in a 10-layer MLP at initialisation

A 10-layer MLP (3072 → 256 → … → 256 → 10, i.e. ten `Linear` layers) is initialised and one backward pass is run on 512 training images. We record $\|\partial L / \partial W_\ell\|_F$ for every layer $\ell$, with sigmoid and with ReLU activations. Each activation is shown with its matching initialisation (Xavier for sigmoid, Kaiming for ReLU) and, dashed, with the other one, to separate the effect of the activation from that of the initialisation.

The sigmoid needs its own backward pass: $\sigma'(x) = \sigma(x)\,(1 - \sigma(x))$.

In [ ]:
class Sigmoid:
    def forward(self, x):
        self.out = torch.sigmoid(x)
        return self.out

    def backward(self, dout):
        return dout * self.out * (1 - self.out)


def layer_grad_norms(activation, init, depth=10, width=256):
    set_seed()
    mlp = MLP([3072] + [width] * (depth - 1) + [10], activation, init)
    mlp.loss(X_train[:512].flatten(1), y_train[:512])
    mlp.backward()
    return [lin.dW.norm().item() for lin in mlp.linears]


runs = {
    "sigmoid, Xavier init": (Sigmoid, "xavier", "#2a78d6", "-"),
    "ReLU, Kaiming init": (ReLU, "kaiming", "#eb6834", "-"),
    "sigmoid, Kaiming init": (Sigmoid, "kaiming", "#2a78d6", "--"),
    "ReLU, Xavier init": (ReLU, "xavier", "#eb6834", "--"),
}

layers = np.arange(1, 11)
plt.figure(figsize=(7, 4.2))
for name, (act, init, color, style) in runs.items():
    norms = layer_grad_norms(act, init)
    plt.semilogy(layers, norms, style, color=color, marker="o", ms=4, label=name)
    print(f"{name:<22} layer 1: {norms[0]:.1e}   layer 10: {norms[-1]:.1e}   ratio: {norms[0] / norms[-1]:.1e}")
plt.xticks(layers)
plt.xlabel("layer (1 = next to the input)")
plt.ylabel(r"$\|\partial L / \partial W_\ell\|_F$")
plt.title("Gradient norm per layer at initialisation")
plt.legend()
plt.show()

_INTERPRETATION 1D_